<a href="https://colab.research.google.com/github/sumaurya/master-data-research/blob/develop/EDA_and_Preprocessing_with_least_dataloss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- read the raw data into a dataframe from merged_master_data.csv

In [1]:
# prompt: read the raw data into a dataframe from merged_master_data.csv

import pandas as pd
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

# Assuming merged_master_data.csv is in the current working directory
original_df = pd.read_csv('merged_master_data.csv')

- drop unwanted columns

In [2]:
# prompt: merge these list.. these are unwanted ciolumns.. drop them
# # skipped 'date' column for now
object_column_names = ['date', 'created_at', 'updated_at', 'annual_earning_date', 'balance_sheet_filing_date', 'cash_flow_filing_date',
                       'earning_history_date', 'exchange_code', 'income_statement_date', 'income_statement_filing_date',
                       'outstanding_shares_date', 'outstanding_shares_date_formatted', 'report_date', 'technical_master_data_sync_date']
int_column_names = ['id', 'hash_code_value', 'exchange', 'fund', 'vendor', 'vendor_api']

unwanted_columns = object_column_names + int_column_names
modified_df = original_df.drop(columns=unwanted_columns, errors='ignore')


print("Original DataFrame shape:", original_df.shape)
print("Modified DataFrame shape:", modified_df.shape)

Original DataFrame shape: (3585, 245)
Modified DataFrame shape: (3585, 225)


In [3]:
# prompt: - create a function in which for each column except 'date' column, create column_name+ "_missing" and set it to True if the values was missing/NaN and False if value is present. return the modified dataframe

def create_missing_indicator_columns(df):
  """
  Creates new columns indicating missing values for each column except 'date'.

  Args:
    df: The input pandas DataFrame.

  Returns:
    The modified DataFrame with new missing indicator columns.
  """

  for column in df.columns:
    if column != 'date':
      df[column + '_missing'] = df[column].isnull()

  return df

# Apply the function to the modified_df
modified_df = create_missing_indicator_columns(modified_df)
print("Modified DataFrame shape:", modified_df.shape)

Modified DataFrame shape: (3585, 450)


- split the dataset into train and test

In [4]:
# prompt: I haven't decided any target variable.. I wanto split the data set as it is into train and test
# - split the dataset into train and test

from sklearn.model_selection import train_test_split

# Split the dataset into train and test sets (e.g., 80% train, 20% test)
train_df, test_df = train_test_split(modified_df, test_size=0.2, random_state=42)

print("Train DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)

Train DataFrame shape: (2868, 450)
Test DataFrame shape: (717, 450)


- create a function in which remove duplicate rows from train dataset. return the modified dataframe

In [5]:
# prompt: - create a function in which remove duplicate rows from train dataset. return the modified dataframe

def remove_duplicate_rows(df):
  """
  Removes duplicate rows from a DataFrame.

  Args:
    df: The input pandas DataFrame.

  Returns:
    The modified DataFrame with duplicate rows removed.
  """
  df = df.drop_duplicates()
  return df

# Example usage:
train_df = remove_duplicate_rows(train_df)
print("Train DataFrame shape after removing duplicates:", train_df.shape)

Train DataFrame shape after removing duplicates: (2868, 450)


- create a function in which. and return the modified dataframe
    - for each numeric column in train dataset :
        - apply mutation and fill NaN with median of the same numeric column. if all values in the column are NaN and median can't be calculated, use median 0. keep the non-null values as it is.
        - also store the calculated median should be stored in median_mutation.csv as column_name,meadian_value

In [6]:
# prompt: - create a function in which. and return the modified dataframe
#     - for each numeric column in train dataset :
#         - apply mutation and fill NaN with median of the same numeric column. if all values in the column are NaN and median can't be calculated, use median 0. keep the non-null values as it is.
#         - also store the calculated median should be stored in median_mutation.csv as column_name,meadian_value

def fill_na_with_median_and_store(df):
  """
  Fills NaN values in numeric columns with the median of the column,
  stores the calculated medians in a CSV file, and returns the modified DataFrame.

  Args:
    df: The input pandas DataFrame.

  Returns:
    The modified DataFrame with NaN values filled.
  """
  median_values = {}
  for column in df.select_dtypes(include=['number']):
    median = df[column].median()
    if pd.isnull(median):
      median = 0
    df[column] = df[column].fillna(median)
    median_values[column] = median

  # Store the median values in median_mutation.csv
  median_df = pd.DataFrame(list(median_values.items()), columns=['column_name', 'median_value'])
  median_df.to_csv('median_mutation.csv', index=False)

  return df

# Apply the function to the train_df
train_df = fill_na_with_median_and_store(train_df)
print("Train DataFrame shape after filling NaN with median:", train_df.shape)

Train DataFrame shape after filling NaN with median: (2868, 450)


- create a function in which. and return the modified dataframe
    - for each numeric column in train dataset :
        - check if the distribution is normal using Shapiro-Wilk Test or not and do following action :
            - if normal distribution,
                - if value in current cell is 2 standard deviation away from the mean, then consider it an outlier and replace with a median
                - store (column_name, 2 standard deviation value, median) in outlier_replacement_normal.csv
            - if not normal distribution,
                - do following
                lower_percentile = df['column'].quantile(0.25)
                upper_percentile = df['column'].quantile(0.75)
                df.loc[df['column'] < lower_bound, 'column'] = lower_percentile
                df.loc[df['column'] > upper_bound, 'column'] = upper_percentile
                - store (column_name, lower_percentile value, upper_percentile value) in outlier_replacement_non_normal.csv

In [7]:
# prompt: - create a function in which. and return the modified dataframe
#     - for each numeric column in train dataset :
#         - check if the distribution is normal using Shapiro-Wilk Test or not and do following action :
#             - if normal distribution,
#                 - if value in current cell is 2 standard deviation away from the mean, then consider it an outlier and replace with a median
#                 - store (column_name, 2 standard deviation value, median) in outlier_replacement_normal.csv
#             - if not normal distribution,
#                 - do following
#                 lower_percentile = df['column'].quantile(0.25)
#                 upper_percentile = df['column'].quantile(0.75)
#                 df.loc[df['column'] < lower_bound, 'column'] = lower_percentile
#                 df.loc[df['column'] > upper_bound, 'column'] = upper_percentile
#                 - store (column_name, lower_percentile value, upper_percentile value) in outlier_replacement_non_normal.csv

from scipy.stats import shapiro

def handle_outliers(df):
  """
  Handles outliers in numeric columns based on their distribution (normal or not).

  Args:
    df: The input pandas DataFrame.

  Returns:
    The modified DataFrame with outliers handled.
  """
  outlier_replacement_normal = []
  outlier_replacement_non_normal = []

  for column in df.select_dtypes(include=['number']):
    try:
      # Check for normality using Shapiro-Wilk Test
      stat, p = shapiro(df[column].dropna())
      if p > 0.05:  # Distribution is considered normal
        mean = df[column].mean()
        std = df[column].std()
        upper_bound = mean + 2 * std
        lower_bound = mean - 2 * std
        median = df[column].median()

        df.loc[(df[column] > upper_bound) | (df[column] < lower_bound), column] = median
        outlier_replacement_normal.append((column, 2 * std, median))

      else:  # Distribution is not normal
        lower_percentile = df[column].quantile(0.25)
        upper_percentile = df[column].quantile(0.75)
        df.loc[df[column] < lower_percentile, column] = lower_percentile
        df.loc[df[column] > upper_percentile, column] = upper_percentile
        outlier_replacement_non_normal.append((column, lower_percentile, upper_percentile))
    except Exception as e:
        print(f"Error processing column {column}: {e}")

  # Store outlier replacement information in CSV files
  pd.DataFrame(outlier_replacement_normal, columns=['column_name', '2_std_value', 'median']).to_csv(
      'outlier_replacement_normal.csv', index=False)
  pd.DataFrame(outlier_replacement_non_normal, columns=['column_name', 'lower_percentile', 'upper_percentile']).to_csv(
      'outlier_replacement_non_normal.csv', index=False)

  return df


# Apply the function to the train_df
train_df = handle_outliers(train_df)
print("Train DataFrame shape after handling outliers:", train_df.shape)

Train DataFrame shape after handling outliers: (2868, 450)


- create a function in which. and return the modified dataframe
    - for each column_name in outlier_replacement_normal.csv
        - apply Standardization
    - for each column_name in outlier_replacement_non_normal.csv
        - apply Normalization
    - store the scaler objects in seperate files using joblib

In [8]:
# prompt: - create a function in which. and return the modified dataframe
#     - for each column_name in outlier_replacement_normal.csv
#         - apply Standardization
#     - for each column_name in outlier_replacement_non_normal.csv
#         - apply Normalization
# do it like
# scaler = StandardScaler()
# scaler.fit(train_data[['feature1', 'feature2', ...]])
# not one column at a time

from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

def apply_scaling(df):
  """
  Applies standardization to columns specified in 'outlier_replacement_normal.csv'
  and normalization to columns specified in 'outlier_replacement_non_normal.csv'.

  Args:
    df: The input pandas DataFrame.

  Returns:
    The modified DataFrame with scaling applied.
  """
  try:
    normal_columns = pd.read_csv('outlier_replacement_normal.csv')['column_name'].tolist()
    non_normal_columns = pd.read_csv('outlier_replacement_non_normal.csv')['column_name'].tolist()

    if normal_columns:
      scaler = StandardScaler()
      df[normal_columns] = scaler.fit_transform(df[normal_columns])
      joblib.dump(scaler, 'scaler_normal.joblib')

    if non_normal_columns:
      scaler = MinMaxScaler()
      df[non_normal_columns] = scaler.fit_transform(df[non_normal_columns])
      joblib.dump(scaler, 'scaler_non_normal.joblib')

    return df

  except FileNotFoundError:
    print("Error: 'outlier_replacement_normal.csv' or 'outlier_replacement_non_normal.csv' not found.")
    return df


# Apply the function to the train_df
train_df = apply_scaling(train_df)
print("Train DataFrame shape after scaling:", train_df.shape)

Train DataFrame shape after scaling: (2868, 450)


- Drop minimum features to remove correlation

In [9]:
def find_unique_paths(adjacency_list):
    def dfs(node, visited, path):
        visited.add(node)
        path.append(node)

        # Record the complete path
        current_path = tuple(path)
        if current_path not in all_paths:
            all_paths[current_path] = set()  # Initialize set to keep track of unique nodes

        # Explore neighbors
        for neighbor in adjacency_list[node]:
            if neighbor not in visited:
                dfs(neighbor, visited, path)

        # Backtrack
        visited.remove(node)
        path.pop()

    all_paths = {}

    # Start DFS from each node
    for start_node in adjacency_list:
        dfs(start_node, set(), [])

    # Filter out paths that are prefixes of other paths
    unique_paths = []
    for path in all_paths:
        is_prefix = False
        for other_path in all_paths:
            if path != other_path and path in other_path:
                is_prefix = True
                break
        if not is_prefix:
            unique_paths.append(list(path))

    return unique_paths

# Example usage:
# adjacency_list = {'A': ['B', 'C', 'D'], 'B': [], 'C': ['D'], 'D': []}
# unique_paths = find_unique_paths(adjacency_list)
# for path in unique_paths:
#     print(" -> ".join(path))


In [10]:
import pandas as pd

def remove_columns_and_save(df, columns_to_be_removed, filename='removed_columns_due_to_high_correlation_numeric.csv'):
    # Identify removed columns
    removed_columns = [col for col in columns_to_be_removed if col in df.columns]

    # Create a DataFrame of removed columns for saving
    removed_columns_df = pd.DataFrame(removed_columns, columns=['Removed Columns'])

    # Remove the columns from the DataFrame
    df_cleaned = df.drop(columns=removed_columns, errors='ignore')

    # Save removed columns to a CSV file
    removed_columns_df.to_csv(filename, index=False)

    return df_cleaned

# Example usage:
# df = pd.DataFrame({...})  # Your DataFrame here
# columns_to_remove = {'column1', 'column2', ...}  # Set of columns to remove
# df_cleaned = remove_columns_and_save(df, columns_to_remove)


In [11]:
import pandas as pd

def drop_high_correlation_columns(df, target_variables, threshold=0.7):
    """
    Drops columns from the DataFrame that are highly correlated with others, ignoring target variables.

    Parameters:
    df (pd.DataFrame): Input DataFrame from which to drop columns.
    target_variables (list): List of target variable names to ignore.
    threshold (float): Correlation threshold for dropping columns.

    Returns:
    pd.DataFrame: DataFrame with reduced columns.
    """

    # Step 1: Filter out target variables and non-numeric features
    numeric_df = df.select_dtypes(include='number').drop(columns=target_variables, errors='ignore')

    # Step 2: Calculate the correlation matrix
    correlation_matrix = numeric_df.corr().abs()  # Use absolute values for correlation

    high_correlation_pairs = []

    for i in range(len(correlation_matrix.columns)):
        for j in range(i):
            if correlation_matrix.iloc[i, j] > threshold:
                col1 = correlation_matrix.columns[i]
                col2 = correlation_matrix.columns[j]
                high_correlation_pairs.append((col1, col2, correlation_matrix.iloc[i, j]))

    adjacency_list = {}

    for col1, col2, _ in high_correlation_pairs:
        if col1 not in adjacency_list:
            adjacency_list[col1] = []
        if col2 not in adjacency_list:
            adjacency_list[col2] = []

        # Add the edges in both directions since it's undirected
        adjacency_list[col1].append(col2)
        adjacency_list[col2].append(col1)

    columns_to_be_removed = set()

    print(adjacency_list);
    unique_paths = find_unique_paths(adjacency_list)
    for path in unique_paths:
      print("->".join(path))
      updated_paths = []
      for i in range(1, len(path), 2):  # Start from index 1 and step by 2
        columns_to_be_removed.add(path[i])
        updated_path = [element for element in path if element not in columns_to_be_removed]
        updated_paths.append(updated_path)
      unique_paths = updated_paths

    df_reduced = remove_columns_and_save(df, columns_to_be_removed)
    return df_reduced

In [12]:
target_variables = ['five_percent_reached_in_one_day', 'days_taken_for_5_percent', 'percentage_change_in_next_open']

In [ ]:
train_df = drop_high_correlation_columns(train_df, target_variables)
print("Train DataFrame shape after removing high correlation columns:", train_df.shape)

{'adjusted_close': ['accounts_payable', 'annual_earning_eps_actual', 'avgvolccy', 'begin_period_cash_flow', 'capital_expenditures', 'cash', 'cash_and_equivalents', 'cash_and_short_term_investments', 'cash_flow_net_income', 'close', 'common_stock', 'common_stock_total_equity', 'cost_of_revenue', 'depreciation', 'depreciation_and_amortization', 'ebit', 'ebitda', 'ema', 'end_period_cash_flow', 'eps_actual', 'free_cash_flow', 'gross_profit', 'high', 'income_before_tax', 'income_tax_expense', 'interest_expense', 'inventory', 'lband', 'liabilities_and_stockholders_equity', 'low', 'mband', 'net_income', 'net_income_applicable_to_common_shares', 'net_receivables', 'non_current_assets_total', 'non_current_liabilities_total', 'non_current_assets_other', 'open', 'operating_income', 'other_current_assets', 'other_current_liabilities', 'other_operating_expenses', 'property_plant_and_equipment_net', 'property_plant_equipment', 'reconciled_depreciation', 'research_development', 'retained_earnings', '